## `Define Necessaray Libraries`

In [97]:
# Import Necessary Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('dark_background')


# Import preprocessing libraries
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer




In [98]:
# Load the dataset  

train_data = pd.read_csv('../data/raw/train.csv')
print('Train Data Shape:', train_data.shape)
train_data.head()

Train Data Shape: (8693, 14)


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [99]:
# Split the dataset into features and target variable
X = train_data.drop(columns=['Transported'], errors='ignore')
y = train_data['Transported']

print('Features Shape:', X.shape)
print('Target Shape:', y.shape)

Features Shape: (8693, 13)
Target Shape: (8693,)


## `Feature Engineering`

In [100]:
X[["Cabin", "PassengerId", "Name"]].sample(10)

,Cabin,PassengerId,Name
2133,F/465/P,2289_01,Hary Jaff
7217,B/253/P,7710_01,Chabih Eguing
2037,F/428/S,2180_01,Coreen Masquez
554,F/109/S,0583_01,Muebix Erle
4977,B/204/S,5307_01,Mirark Fuelisent
2277,F/507/P,2446_01,Lestie Hoppernardy
8091,F/1668/S,8644_01,Shawne Dunnisey
4105,B/169/S,4383_01,Sheratz Dingauge
7599,C/302/S,8122_01,Sadiram Coning
5933,C/192/P,6299_01,Merops Femoused


In [101]:
# Split the Cabin column into Deck, CabinNumber and Side

X[['Deck', 'CabinNumber', 'Side']] = X['Cabin'].str.split('/', expand=True)
X["CabinNumber"] = pd.to_numeric(X["CabinNumber"], errors='coerce')


In [102]:
X["GroupNumber"] = X["PassengerId"].str.split('_').str[1]
X["GroupNumber"] = pd.to_numeric(X["GroupNumber"], errors='coerce')

group_counts = X["GroupNumber"].value_counts()
X["GroupSize"] = X["GroupNumber"].map(group_counts)


In [103]:
# Drop redundant columns
X.drop(columns=['PassengerId', 'Name', 'Cabin', 'GroupNumber'], inplace=True, errors='ignore')

In [104]:
spending_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

X["TotalSpending"] = X[spending_cols].sum(axis=1)

In [105]:
X.shape

(8693, 15)

In [106]:
X.head(10)

,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Deck,CabinNumber,Side,GroupSize,TotalSpending
0,Europa,False,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,B,0.0,P,6217,0.0
1,Earth,False,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,F,0.0,S,6217,736.0
2,Europa,False,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,A,0.0,S,6217,10383.0
3,Europa,False,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,A,0.0,S,1412,5176.0
4,Earth,False,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,F,1.0,S,6217,1091.0
5,Earth,False,PSO J318.5-22,44.0,False,0.0,483.0,0.0,291.0,0.0,F,0.0,P,6217,774.0
6,Earth,False,TRAPPIST-1e,26.0,False,42.0,1539.0,3.0,0.0,0.0,F,2.0,S,6217,1584.0
7,Earth,True,TRAPPIST-1e,28.0,False,0.0,0.0,0.0,0.0,NaN,G,0.0,S,1412,0.0
8,Earth,False,TRAPPIST-1e,35.0,False,0.0,785.0,17.0,216.0,0.0,F,3.0,S,6217,1018.0
9,Europa,True,55 Cancri e,14.0,False,0.0,0.0,0.0,0.0,0.0,B,1.0,P,6217,0.0


In [107]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 15 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   HomePlanet     8492 non-null   str    
 1   CryoSleep      8476 non-null   object 
 2   Destination    8511 non-null   str    
 3   Age            8514 non-null   float64
 4   VIP            8490 non-null   object 
 5   RoomService    8512 non-null   float64
 6   FoodCourt      8510 non-null   float64
 7   ShoppingMall   8485 non-null   float64
 8   Spa            8510 non-null   float64
 9   VRDeck         8505 non-null   float64
 10  Deck           8494 non-null   str    
 11  CabinNumber    8494 non-null   float64
 12  Side           8494 non-null   str    
 13  GroupSize      8693 non-null   int64  
 14  TotalSpending  8693 non-null   float64
dtypes: float64(8), int64(1), object(2), str(4)
memory usage: 1018.8+ KB


In [108]:
# Split columns into numerical and categorical features
numerical_features = X_train.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X_train.select_dtypes(
    exclude=np.number
).columns.tolist()

print(f"Total length of num + cat columns: {len(Categorical_cols) + len(Numerical_cols)}")
print(f"Numerical features: {len(numerical_features)}")
print(f"Categorical features: {len(categorical_features)}")

print(numerical_features)
print(categorical_features)

Total length of num + cat columns: 15
Numerical features: 9
Categorical features: 6
['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'CabinNumber', 'GroupSize', 'TotalSpending']
['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Deck', 'Side']


## `Train-validation split`

In [109]:
X_train, X_val, y_train, y_val = train_test_split(X, 
                                                    y, 
                                                    test_size=0.2, 
                                                    random_state=42, 
                                                    stratify=y)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)

X_train: (6954, 15)
X_val: (1739, 15)
y_train: (6954,)
y_val: (1739,)


## `Deifine pipelines`

In [110]:
# Numerical preprocessing pipeline
numerical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])


# Combine the numerical and categorical pipelines into a single preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('numerical', 
        numerical_pipeline, 
        numerical_features),

        ('categorical', 
        categorical_pipeline, 
        categorical_features)
    ]
)

## `Fit and train`

In [111]:
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)

In [112]:
# Before processing, the shape of the data is:
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)

# After processing, the shape of the data is:
print("X_train_processed:", X_train_processed.shape)
print("X_val_processed:", X_val_processed.shape)

X_train: (6954, 15)
X_val: (1739, 15)
X_train_processed: (6954, 29)
X_val_processed: (1739, 29)


In [117]:
assert X_train_processed.shape[0] == len(y_train)
assert X_val_processed.shape[0] == len(y_val)
assert np.isnan(X_train_processed).sum() == 0
assert np.isnan(X_val_processed).sum() == 0

print("All preprocessing checks passed.")

All preprocessing checks passed.
